In [21]:
import pandas as pd
import time
import warnings
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
warnings.filterwarnings("ignore")
from datetime import date
import re
from collections import defaultdict

In [ ]:
str(date.today())
# ''.join(str(date.today()).split('-'))

'2025-08-11'

In [25]:
# 스크롤 다운 함수
def scroll_down(driver, max_scroll=10):
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scroll):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


match_url = "https://sports.daum.net/schedule/kbo"
driver = webdriver.Chrome()
driver.get(match_url)
wait = WebDriverWait(driver, 3)

# 1. 캘린더 버튼 선택
calander_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'btn_calendar')))
driver.execute_script("arguments[0].click()", calander_button)

# 2. 년/월 버튼 선택
# 1) 년 버튼 펼치기 + 선택
expand_button_y = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#hermesData > div.box_schedule.schedule_kleague > div.calendar_g > div.info_calendar.on > div > div.head_calendar > div.choice_calendar.choice_year > a > span')))
expand_button_y.click()
year_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, f"a[data-year='{date.today().year}']")))
driver.execute_script("arguments[0].click()", year_button)
# 2) 월 버튼 펼치기 + 선택
expand_button_m = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#hermesData > div.box_schedule.schedule_kleague > div.calendar_g > div.info_calendar.on > div > div.head_calendar > div.choice_calendar.choice_month > a > span')))
driver.execute_script("arguments[0].click()", expand_button_m)
month_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, f"a[data-month='{date.today().month:02d}']")))
driver.execute_script("arguments[0].click()", month_button)

# 3. 스크롤 다운 
scroll_down(driver, max_scroll=10)

# 4. 해당 년, 월의 경기 일정 모두 가져오기
# 날짜별 경기 데이터를 저장할 딕셔너리
schedule_by_date = defaultdict(list)
current_date = None
current_game = []
is_game_block = False

# 날짜 패턴을 정규식으로 정의
date_pattern = re.compile(r'^\d{2}\.\d{2}')

sched = wait.until(EC.presence_of_all_elements_located((By.ID, 'scheduleList')))
for sh in sched:
    text_lines = sh.text.strip().split('\n')
    for i, line in enumerate(text_lines):
        if date_pattern.match(line):
            if current_game:
                schedule_by_date[current_date].append(', '.join(current_game))
                current_game = []
            
            current_date_str = ' '.join(line.split()[:2])  # '08.01 금'
            current_date = current_date_str

            if "경기가 없습니다." in line:
                schedule_by_date[current_date].append("경기가 없습니다.")
                current_date = None  
                is_game_block = False
            else:
                is_game_block = True
                current_game.append(' '.join(line.split()[2:]))

        elif is_game_block:
            if re.match(r'^\d{2}:\d{2}', line) and current_game:
                schedule_by_date[current_date].append(', '.join(current_game))
                current_game = [line]
            else:
                current_game.append(line)

    # 마지막 경기 데이터를 추가
    if current_game and current_date:
        schedule_by_date[current_date].append(', '.join(current_game))

# 결과 출력
import json
print(json.dumps(schedule_by_date, indent=2, ensure_ascii=False))

{
  "08.01 금": [
    "18:30 광주-기아 챔피언스필드, 종료, 한화, 2, KIA, 3, 페넌트레이스 SPOTV, 하이라이트보러가기",
    "18:30 대구 삼성 라이온즈 파크, 종료, LG, 4, 삼성, 2, 페넌트레이스 SPOTV2, 하이라이트보러가기",
    "18:30 고척 스카이돔, 종료, 롯데, 0, 키움, 2, 페넌트레이스 KBS N SPORTS, 하이라이트보러가기",
    "18:30 잠실야구장, 종료, SSG, 7, 두산, 2, 페넌트레이스 SBS SPORTS, 하이라이트보러가기",
    "18:30 창원NC파크, 종료, KT, 3, NC, 5, 페넌트레이스 MBC SPORTS+, 하이라이트보러가기"
  ],
  "08.02 토": [
    "18:00 광주-기아 챔피언스필드, 경기취소, 한화, -, KIA, -, 페넌트레이스 SPOTV, 경기취소",
    "18:00 대구 삼성 라이온즈 파크, 종료, LG, 3, 삼성, 2, 페넌트레이스 SPOTV2, 하이라이트보러가기",
    "18:00 고척 스카이돔, 종료, 롯데, 3, 키움, 2, 페넌트레이스 KBS N SPORTS, 하이라이트보러가기",
    "18:00 잠실야구장, 종료, SSG, 4, 두산, 5, 페넌트레이스 SBS SPORTS, 하이라이트보러가기",
    "18:00 창원NC파크, 종료, KT, 6, NC, 7, 페넌트레이스 MBC SPORTS+, 하이라이트보러가기"
  ],
  "08.03 일": [
    "14:00 고척 스카이돔, 종료, 롯데, 9, 키움, 3, 페넌트레이스 KBS N SPORTS, 하이라이트보러가기",
    "18:00 광주-기아 챔피언스필드, 경기취소, 한화, -, KIA, -, 페넌트레이스 SPOTV, 경기취소",
    "18:00 대구 삼성 라이온즈 파크, 종료, LG, 6, 삼성, 3, 페넌트레이스 SPOTV2, 하이라이트보러가기",
    "18:00 잠실야구장, 종료, SSG, 3, 두산, 2, 페넌트레

In [44]:
for date, match in schedule_by_date.items():
    print(date)
    if len(match) == 1:
        print('경기가 없습니다')
    else:
        for m in match:
            # print(m.split(','))
            print('시간:', m.split(',')[0].split(' ')[0])
            print('구장:', m.split(',')[0].split(' ')[1])
            print('원정팀:', m.split(',')[2], m.split(',')[3])
            print('홈팀:', m.split(',')[4], m.split(',')[5])
    print("--------------------------------------------------")


08.01 금
시간: 18:30
구장: 광주-기아
원정팀:  한화  2
홈팀:  KIA  3
시간: 18:30
구장: 대구
원정팀:  LG  4
홈팀:  삼성  2
시간: 18:30
구장: 고척
원정팀:  롯데  0
홈팀:  키움  2
시간: 18:30
구장: 잠실야구장
원정팀:  SSG  7
홈팀:  두산  2
시간: 18:30
구장: 창원NC파크
원정팀:  KT  3
홈팀:  NC  5
--------------------------------------------------
08.02 토
시간: 18:00
구장: 광주-기아
원정팀:  한화  -
홈팀:  KIA  -
시간: 18:00
구장: 대구
원정팀:  LG  3
홈팀:  삼성  2
시간: 18:00
구장: 고척
원정팀:  롯데  3
홈팀:  키움  2
시간: 18:00
구장: 잠실야구장
원정팀:  SSG  4
홈팀:  두산  5
시간: 18:00
구장: 창원NC파크
원정팀:  KT  6
홈팀:  NC  7
--------------------------------------------------
08.03 일
시간: 14:00
구장: 고척
원정팀:  롯데  9
홈팀:  키움  3
시간: 18:00
구장: 광주-기아
원정팀:  한화  -
홈팀:  KIA  -
시간: 18:00
구장: 대구
원정팀:  LG  6
홈팀:  삼성  3
시간: 18:00
구장: 잠실야구장
원정팀:  SSG  3
홈팀:  두산  2
시간: 18:00
구장: 창원NC파크
원정팀:  KT  1
홈팀:  NC  1
--------------------------------------------------
08.04 월
경기가 없습니다
--------------------------------------------------
08.05 화
시간: 18:30
구장: 사직야구장
원정팀:  KIA  2
홈팀:  롯데  0
시간: 18:30
구장: 대전
원정팀:  KT  5
홈팀:  한화  2
시간: 18:30
구장: 잠실야구장
원정팀:  두